In [1]:
# ############################################################
# 다중분류 예시 — 3등급 분류 & macro vs weighted f1
# ############################################################
# ------------------------------------------------------------
# [목적] 도구 불러오기 — 데이터 / 분리 / 모델 / 평가
# ------------------------------------------------------------
import numpy as np                                # 숫자 계산 (개수 세기)
from sklearn.datasets import make_classification  # 가짜 분류 데이터 생성기 (연습용 데이터 만들기)
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier    # 랜덤포레스트 (나무들의 다수결)
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [2]:
# ------------------------------------------------------------
# [목적] 예시용 '불균형' 3등급 데이터 (Poor/Standard/Good 흉내)
#        (Standard가 많은 불균형 -> macro/weighted 차이를 보기 좋음)
# ------------------------------------------------------------
X, y = make_classification(n_samples=600, n_features=6,   # 600개, 특성 6개
                           n_informative=4, n_redundant=0,# 쓸모있는 특성 4개
                           n_classes=3,                   # 3등급 (0/1/2)
                           weights=[0.2, 0.6, 0.2],       # 등급 비율(불균형: 가운데가 많음)
                           random_state=0)                # 결과 고정
print('등급별 개수:', np.bincount(y))             # 각 등급이 몇 개인지 (불균형 확인)

등급별 개수: [123 358 119]


In [3]:
# ------------------------------------------------------------
# [목적] 8:2 분리 -> 학습 -> 예측
# ------------------------------------------------------------
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y)      # 등급 비율 유지하며 8:2
rf = RandomForestClassifier(n_estimators=100, random_state=0).fit(X_tr, y_tr)  # 학습
pred = rf.predict(X_val)                          # 예측 (각 샘플의 등급)

In [4]:
# ------------------------------------------------------------
# [목적] 채점 — 다중분류는 average를 반드시 지정!
# ------------------------------------------------------------
print('정확도       :', round(accuracy_score(y_val, pred), 3))                 # 전체 맞힌 비율
print('f1(macro)    :', round(f1_score(y_val, pred, average='macro'), 3))      # 등급을 '똑같이' 평균
print('f1(weighted) :', round(f1_score(y_val, pred, average='weighted'), 3))   # 등급을 '개수로 가중' 평균
print(classification_report(y_val, pred))         # 등급별 정밀도·재현율·f1 한눈에

정확도       : 0.892
f1(macro)    : 0.858
f1(weighted) : 0.89


              precision    recall  f1-score   support

           0       0.86      0.79      0.83        24
           1       0.92      0.96      0.94        72
           2       0.83      0.79      0.81        24

    accuracy                           0.89       120
   macro avg       0.87      0.85      0.86       120
weighted avg       0.89      0.89      0.89       120



In [5]:
# ============================================================
# [결과 해석]
#  · 등급별 개수 [123, 358, 119] -> 가운데(Standard)가 많은 '불균형'
#  · 정확도 ~ 0.89
#  · f1(macro) ~ 0.858  <  f1(weighted) ~ 0.890  ← 핵심!
#      - macro   : 세 등급을 똑같이 평균 (소수 등급 성적이 낮으면 확 떨어짐)
#      - weighted: 개수로 가중 평균 (다수 등급 성적이 크게 반영)
#  · 불균형일수록 둘을 함께 봐야 소수 등급을 놓치지 않음
# ============================================================